In [ ]:
set.seed(2026)
library(tidyverse)
library(rsample)

# Read in data
# getwd()
# file_url <- "https://data.nasa.gov/docs/legacy/meteorite_landings/Meteorite_Landings.csv"
# dest_file <- "../data/raw/meteorite_landings.csv"
# download.file(file_url, dest_file, method = "auto")
meteors <- read.csv("../data/raw/meteorite_landings.csv")
# head(meteors)

# Clean data
meteors_clean <- meteors %>%
  drop_na(mass..g., year, reclat, reclong, fall, recclass) %>%
  mutate(
    mass = as.numeric(mass..g.),
    year = as.numeric(year),
    reclat = as.numeric(reclat),
    reclong = as.numeric(reclong)
  ) %>%
  drop_na(mass, year, reclat, reclong) %>%
  filter(mass > 0) %>%
  mutate(
    log_mass = log10(mass),
    fall = as.factor(fall),
    recclass = as.factor(recclass)
  )

top_classes <- meteors_clean %>%
  count(recclass, sort = TRUE) %>%
  slice_head(n = 5) %>%
  pull(recclass)

meteors_clean <- meteors_clean %>%
  filter(recclass %in% top_classes)


# Split data into training/testing
split_obj <- initial_split(meteors_clean, prop = 0.8)
train_data <- training(split_obj)
test_data  <- testing(split_obj)
head(test_data)

# Linear Regression model
model <- glm(
  log_mass ~ year * recclass + reclat + reclong + fall,
  data = train_data,
  family = gaussian()
)
tidy(model)


# Predictions

year_seq <- seq(min(meteors_clean$year), max(meteors_clean$year), length.out = 100)

predictions <- predict(model, newdata = test_data)
prediction_grid <- expand.grid(
  year = year_seq,
  reclat = mean(meteors_clean$reclat),
  reclong = mean(meteors_clean$reclong),
  fall = levels(meteors_clean$fall)[1],   # fix one category
  recclass = top_classes
)

prediction_grid$pred_log_mass <- predict(model, newdata = prediction_grid)

# Plot
plot_classes <- ggplot(meteors_clean, aes(x = year, y = log_mass, color = recclass)) +
  geom_point(alpha = 0.3) +
  geom_line(
    data = prediction_grid,
    aes(x = year, y = pred_log_mass, color = recclass),
    linewidth = 1.2
  ) +
  labs(
    title = "Regression Lines by Meteorite Class",
    x = "Year",
    y = "Log10(Mass)",
    color = "Meteorite Class"
  ) +
  theme_minimal()

plot <- ggplot(test_data, aes(x = log_mass, y = predictions)) +
  geom_point(alpha = 0.4) +
  geom_abline(intercept = 0, slope = 1, color = "red") +
  labs(
    title = "Predicted vs Actual Log(Mass)",
    x = "Actual Log10(Mass)",
    y = "Predicted Log10(Mass)"
  ) +
  theme_minimal()

ggsave(
  filename = "../output/regression_by_class.png",
  plot = plot_classes,
  width = 9,
  height = 6,
  dpi = 300
)

ggsave(
  filename = "../output/predicted_vs_actual.png",
  plot = plot,
  width = 9,
  height = 6,
  dpi = 300
)

,name,id,nametype,recclass,mass..g.,fall,year,reclat,reclong,GeoLocation,mass,log_mass
,<chr>,<int>,<chr>,<fct>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>
1,Aachen,1,Valid,L5,21,Fell,1880,50.77500,6.08333,"(50.775, 6.08333)",21,1.322219
2,Aarhus,2,Valid,H6,720,Fell,1951,56.18333,10.23333,"(56.18333, 10.23333)",720,2.857332
3,Aleppo,462,Valid,L6,3200,Fell,1873,36.23333,37.13333,"(36.23333, 37.13333)",3200,3.505150
4,Ambapur Nagla,2290,Valid,H5,6400,Fell,1895,27.66667,78.25000,"(27.66667, 78.25)",6400,3.806180
5,Andover,2295,Valid,L6,3200,Fell,1898,44.61667,-70.75000,"(44.61667, -70.75)",3200,3.505150
6,Anlong,2305,Valid,H5,2500,Fell,1971,25.15000,105.18333,"(25.15, 105.18333)",2500,3.397940


term,estimate,std.error,statistic,p.value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),2.679944e+01,1.929614e+00,13.8884993,1.191080e-43
year,-1.230529e-02,9.748035e-04,-12.6233548,2.167140e-36
recclassH5,-8.166999e+00,2.271553e+00,-3.5953371,3.247580e-04
recclassH6,-1.083510e+01,2.405525e+00,-4.5042549,6.698617e-06
recclassL5,-9.234538e-01,2.668631e+00,-0.3460403,7.293161e-01
recclassL6,-1.149275e+01,2.083979e+00,-5.5148124,3.535106e-08
reclat,1.056176e-02,1.790829e-04,58.9769312,0.000000e+00
reclong,3.931378e-04,9.802257e-05,4.0106865,6.076429e-05
fallFound,-5.140577e-01,5.122153e-02,-10.0359699,1.206154e-23
